# 03 – 3D Gaussian Splatting for sparse-view CT (3DGS – Gianne)

**Question:** Can a set of optimised 3D Gaussians (here: their 2D slice version) reconstruct a CT slice from only a few projection angles, and how well can it predict projections at angles it has never seen?

**Approach**
1. Load the SinoCT sinograms (`.npy`, 984 angles × 888 detector pixels per slice).
2. Keep only $n$ evenly spaced angles – **exactly the same `view_indices` as the FBP baseline** (246 / 98 / 49 / 25 views).
3. Represent the slice by $N$ anisotropic Gaussians. Initialise them from the sparse-view FBP (same input the other methods get).
4. Optimise position, scale, rotation and density so that the **analytic projections** of the Gaussians match the sparse sinogram (L1 loss + small TV term, adaptive clone / split / prune of Gaussians).
5. Render the 888×888 image → save as float `.npy` in the **raw FBP units** (no 0–1 rescaling).
6. Evaluate with the shared `metrics.py`: (a) image vs. full-view FBP ground truth, (b) **view synthesis** = rendered vs. true projections at held-out angles.

The full-view ground truth is used **only for evaluation**, never for fitting. Each slice is optimised on its own (per-scan optimisation, no training set).

> **Status:** written without being run (a full run takes hours). Work through the sections in order: geometry check (§5) → smoke test (§6) → tuning on the validation series (§7) → full run (§8).
> Tags in the code comments: `TODO(run)` = you must do/check this · `TODO(tune)` = hyper-parameter · `TODO(improve)` = optional upgrade.

## 0. Setup

In [1]:
# Mount Google Drive (data is NOT in git – too large)
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    # get metrics.py from the group repo
    # TODO(run): same clone as Sophia's notebook; if metrics.py has been merged to another branch, change -b
    if not os.path.exists("/content/IPPR_CT_Reconstruction"):
        !git clone -q -b baseline-sophia https://github.com/gs0232/IPPR_CT_Reconstruction.git /content/IPPR_CT_Reconstruction
    sys.path.insert(0, "/content/IPPR_CT_Reconstruction/Baselines")
else:
    sys.path.insert(0, ".")          # locally: metrics.py lies next to this notebook
    sys.path.insert(0, "../Baselines")

In [ ]:
%pip install torch torchvision torchaudio


In [2]:
# TODO(run): Runtime -> Change runtime type -> GPU (T4 is enough). On CPU this is far too slow.
import math, re, time, zipfile
from dataclasses import dataclass, field, replace
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint
from skimage.transform import iradon, radon

import metrics   # our shared metric functions (PSNR, SSIM, RMSE, head mask)
metrics._self_test()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "NO GPU!")

ModuleNotFoundError: No module named 'torch'

## 1. Configuration
Adjust the paths once. The protocol constants must **not** be changed – every method (FBP, U-Net, NeRF, 3DGS) has to see the same angles and geometry.
Tuning knobs are in the `Cfg` class below, each marked `TODO(tune)`.

In [ ]:
# ---- protocol constants: copied from 02_fbp_baseline.ipynb -> DO NOT CHANGE ----
N_ANGLES    = 984          # projection angles in a full SinoCT sinogram
N_DET       = 888          # detector pixels
ANGLE_RANGE = 360          # degrees covered by the 984 angles
OUTPUT_SIZE = 888          # reconstruction size
FILTER      = "ramp"       # FBP filter (only used for the FBP initialisation + ground truth)
VIEW_COUNTS = [246, 98, 49, 25]   # 25 %, 10 %, 5 %, 2.5 % of the 984 angles
SLICE_STEP  = 4            # every 4th slice of each series (same as the baseline)

THETA_FULL_DEG = np.linspace(0, ANGLE_RANGE, N_ANGLES, endpoint=False)


def view_indices(n_views):
    """Same angle selection as the baseline (spread over the first 180 deg only,
    because theta and theta+180 carry the same information)."""
    n_half = N_ANGLES if ANGLE_RANGE == 180 else N_ANGLES // 2
    return np.round(np.linspace(0, n_half, n_views, endpoint=False)).astype(int)

for v in VIEW_COUNTS:
    print(f"{v:4d} views -> one view every {180 / v:.1f}°")

In [ ]:
@dataclass
class Cfg:
    # ---- paths (Colab defaults from the baseline notebook) ------------------------
    data_root: str = "/content/drive/MyDrive/MGST_5_IPPR_CT_Reconstruction"  # TODO(run)
    sino_zip: str = ""       # default: <data_root>/SinoCT_files.zip
    sino_dir: str = "/content/sinograms"
    gt_dir: str = ""         # default: <data_root>/baseline_results/gt_fullview_npy
    out_dir: str = ""        # default: <data_root>/outputs/3dgs   TODO(run): confirm name with Sophia

    # ---- what to run ----------------------------------------------------------------
    # TODO(run): U-Net test series from Jule's log; confirm with Sophia. None = all series.
    series: list = field(default_factory=lambda: ["series_1014", "series_1028"])
    views: list = field(default_factory=lambda: list(VIEW_COUNTS))
    slice_step: int = SLICE_STEP
    max_slices: int = 0      # 0 = all slices of the series (every slice_step-th)
    seed: int = 42

    # ---- model / init -----------------------------------------------------------------
    init: str = "fbp"        # "fbp" | "random"   TODO(improve): ablate in the report
    n_init: int = 15000      # TODO(tune)
    max_gaussians: int = 40000   # TODO(tune) memory/time grow linearly with this
    init_gamma: float = 0.5
    init_uniform_frac: float = 0.1
    s_min: float = 0.5       # px, clamp on Gaussian scale
    s_max: float = 60.0
    angle_sign: float = 1.0  # TODO(run): set to -1 only if --sanity says so

    # ---- optimisation ------------------------------------------------------------------
    iters: int = 3000        # TODO(tune) R2-Gaussian uses 30k in 3D; 2D converges much faster
    views_per_step: int = 8  # TODO(tune) lower on OOM
    gauss_chunk: int = 16384 # TODO(tune) lower on OOM
    use_checkpoint: bool = True
    lr_xy: float = 0.1       # TODO(tune) pixels per step (Adam)
    lr_scale: float = 0.01
    lr_rot: float = 0.01
    lr_rho: float = 0.02
    lr_final_ratio: float = 0.05   # lr decays exponentially to this fraction
    tv_weight: float = 0.01  # TODO(tune) 0 disables TV
    tv_patch: int = 32
    tv_chunk: int = 4096
    l2_weight: float = 0.0   # TODO(tune) optional extra L2 term on the sinogram

    # ---- adaptive density control -----------------------------------------------------
    densify_from: float = 0.10    # fractions of `iters`
    densify_until: float = 0.60
    densify_every: int = 100
    densify_top_frac: float = 0.05
    split_px: float = 4.0         # TODO(tune)
    prune_rho_rel: float = 0.003  # prune if density < this * tissue-density reference

    # ---- evaluation / io ------------------------------------------------------------------
    n_heldout: int = 64      # number of held-out angles for the view-synthesis metrics
    save_preview: bool = True
    log_every: int = 250
    smoke: bool = False
    tag: str = ""            # non-empty -> outputs/CSV go to <out_dir>/_<tag>/ (use for tuning runs!)

    def finalize(self):
        root = Path(self.data_root)
        self.sino_zip = self.sino_zip or str(root / "SinoCT_files.zip")
        self.gt_dir = self.gt_dir or str(root / "baseline_results" / "gt_fullview_npy")
        self.out_dir = self.out_dir or str(root / "outputs" / "3dgs")
        return self

In [ ]:
DATA_ROOT = "/content/drive/MyDrive/MGST_5_IPPR_CT_Reconstruction"   # <- adjust if needed

cfg = Cfg(
    data_root = DATA_ROOT,
    sino_zip  = f"{DATA_ROOT}/SinoCT_files.zip",
    sino_dir  = "/content/sinograms",                      # local Colab disk (fast, does not use Drive space)
    gt_dir    = f"{DATA_ROOT}/baseline_results/gt_fullview_npy",   # Sophia's cached full-view ground truth
    out_dir   = f"{DATA_ROOT}/outputs/3dgs",               # TODO(run): confirm folder name with Sophia
)

# Which series to run. TODO(run): U-Net test series from Jule's log - confirm with Sophia.
# None = all 94 series (about 4000 fits - far too many for the deadline).
cfg.series = ["series_1014", "series_1028"]
cfg.angle_sign = 1.0       # the geometry check (§5) gave +1 (error 0.08 %), keep unless it changes
cfg.finalize()
print("outputs ->", cfg.out_dir)

## 2. Data helpers
Same layout handling as the baseline notebook (`as_slices`, `series_id`).
Only the sinograms of the series in `cfg.series` are extracted from the zip (a few hundred MB on Colab's local disk instead of the whole archive).

In [ ]:
def as_slices(arr):
    """-> (n_slices, N_DET, N_ANGLES). SinoCT files are stored as (984, 888, n_slices)."""
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim == 2:
        arr = arr[None]
    a_ax = [i for i, s in enumerate(arr.shape) if s == N_ANGLES]
    d_ax = [i for i, s in enumerate(arr.shape) if s == N_DET]
    if not a_ax or not d_ax:
        raise ValueError(f"unexpected sinogram shape {arr.shape}")
    s_ax = ({0, 1, 2} - {a_ax[0], d_ax[0]}).pop()
    return np.transpose(arr, (s_ax, d_ax[0], a_ax[0]))


def series_id(path, root):
    m = re.search(r"(\d{3,})", str(Path(path).relative_to(root)))
    return f"series_{m.group(1)}" if m else Path(path).stem


def set_seed(seed):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [ ]:
def find_sinograms(cfg):
    """Return the sinogram .npy files for cfg.series, extracting ONLY those series from the
    zip if they are not on disk yet (saves time and space; works on Windows/Linux/Colab)."""
    sino_dir = Path(cfg.sino_dir)
    wanted = set(cfg.series) if cfg.series else None
    have = {series_id(f, sino_dir) for f in sino_dir.rglob("*.npy")} if sino_dir.exists() else set()
    missing = (not have) if wanted is None else bool(wanted - have)
    if missing:
        zp = Path(cfg.sino_zip)
        if not zp.exists():
            raise SystemExit(
                f"Sinogram zip not found: {zp}\n"
                "TODO(run): set cfg.sino_zip (or cfg.data_root), or point "
                "cfg.sino_dir at a folder that already contains the extracted series.")
        print(f"extracting {sorted(wanted) if wanted else 'ALL series'} from {zp.name} ...")
        with zipfile.ZipFile(zp) as z:
            for name in z.namelist():
                m = re.search(r"(\d{3,})", name)
                sid = f"series_{m.group(1)}" if m else None
                if name.endswith(".npy") and (wanted is None or sid in wanted):
                    z.extract(name, sino_dir)
    files = sorted(sino_dir.rglob("*.npy"))
    if wanted:
        files = [f for f in files if series_id(f, sino_dir) in wanted]
    if not files:
        raise SystemExit("no sinogram files found - check --sino-dir / --sino-zip / --series")
    return files, sino_dir

In [ ]:
def load_gt(cfg, sid, s, sino_slice):
    """Full-view FBP ground truth, cached by Sophia as <sid>_slice<SS>.npy."""
    path = Path(cfg.gt_dir) / f"{sid}_slice{s:02d}.npy"
    if path.exists():
        return np.load(path)
    print(f"  [warn] {path.name} missing -> computing full-view FBP (~14 s). "
          "TODO(run): make sure gt_dir points at Sophia's gt_fullview_npy folder.")
    gt = iradon(sino_slice, theta=THETA_FULL_DEG, filter_name=FILTER,
                output_size=OUTPUT_SIZE, circle=True).astype(np.float32)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.save(path, gt)
    return gt

## 3. The Gaussian model

**Why this is "3DGS for CT" and not RGB splatting:** there is no colour, no alpha compositing and no depth sorting. A CT measurement is a *line integral* of attenuation, so Gaussians simply **add up**. For parallel-beam CT the line integral of a 2D Gaussian along a ray has a closed form (again a 1D Gaussian on the detector), so the differentiable "rasteriser" is exact and plain PyTorch is enough.

Gaussian $i$: centre $\mu_i$, scales $(s_1,s_2)$, rotation $\varphi_i$, peak density $\rho_i$. For projection angle $\theta$:

$$t = x\cos\theta - y\sin\theta,\qquad \sigma_t^2 = s_1^2\cos^2(\theta+\varphi) + s_2^2\sin^2(\theta+\varphi)$$
$$p_i(t) = \rho_i\, 2\pi s_1 s_2\;\mathcal N\!\big(t;\; \mu_i\!\cdot\!(\cos\theta,-\sin\theta),\; \sigma_t^2 + \tfrac1{12}\big),\qquad \text{sinogram}(t,\theta)=\sum_i p_i(t)$$

(the $\tfrac1{12}$ approximates the 1-pixel detector bin).

**Geometry** is taken from `skimage.transform.iradon(circle=True)`, i.e. the same frame as the FBP baseline and the ground truth: pixel (row *i*, col *j*) ↔ $y=i-444,\ x=j-444$; detector bin $k$ ↔ $t=k-444$; zero outside the circle of radius 444; image units = sum along rays with 1 px spacing, so the result lives in the same units as the FBP ground truth and is never rescaled. §5 checks this numerically against `skimage.radon`.

`TODO(improve)`: the dense (views × Gaussians × bins) evaluation is the simple, readable baseline. A tile-based CUDA/Triton kernel touching only bins within ±4σ would be 10–100× faster.

In [ ]:
TWO_PI = 2.0 * math.pi
PIX_VAR = 1.0 / 12.0   # variance of a 1-pixel-wide box -> detector bin integration


def _project_chunk(xy, log_scale, rot, log_rho, theta, det, s_min, s_max):
    """Project a set of Gaussians for several views.

    xy (n,2), log_scale (n,2), rot (n,), log_rho (n,)
    theta (V,) radians, det (D,) detector coordinates in pixels
    returns (V, D) projections.
    """
    s = torch.exp(log_scale).clamp(s_min, s_max)           # (n,2)
    rho = torch.exp(log_rho)                                # (n,)
    s1, s2 = s[:, 0], s[:, 1]

    cos_t = torch.cos(theta)[:, None]                       # (V,1)
    sin_t = torch.sin(theta)[:, None]
    m = xy[:, 0][None, :] * cos_t - xy[:, 1][None, :] * sin_t   # (V,n) centre on detector

    phi = theta[:, None] + rot[None, :]                     # (V,n)
    var = (s1 ** 2)[None, :] * torch.cos(phi) ** 2 \
        + (s2 ** 2)[None, :] * torch.sin(phi) ** 2 + PIX_VAR    # (V,n)
    amp = (rho * TWO_PI * s1 * s2)[None, :] / torch.sqrt(TWO_PI * var)  # (V,n)

    diff = det[None, None, :] - m[:, :, None]               # (V,n,D)
    prof = torch.exp(-0.5 * diff * diff / var[:, :, None])  # (V,n,D)
    return torch.bmm(amp.unsqueeze(1), prof).squeeze(1)     # (V,D) = sum_n amp * prof

In [ ]:
class GaussianCT(nn.Module):
    def __init__(self, xy, log_scale, rot, log_rho, size=888, s_min=0.5, s_max=60.0):
        super().__init__()
        self.xy = nn.Parameter(xy.contiguous().float())               # (N,2)
        self.log_scale = nn.Parameter(log_scale.contiguous().float()) # (N,2)
        self.rot = nn.Parameter(rot.contiguous().float())             # (N,)
        self.log_rho = nn.Parameter(log_rho.contiguous().float())     # (N,)
        self.size = size
        self.c = size // 2
        self.s_min = float(s_min)
        self.s_max = float(s_max)

    # -- convenience --------------------------------------------------------
    @property
    def n(self):
        return self.xy.shape[0]

    def scales(self):
        return torch.exp(self.log_scale).clamp(self.s_min, self.s_max)

    def rho(self):
        return torch.exp(self.log_rho)

    # -- forward model: Gaussians -> sinogram columns -------------------------
    def project(self, theta, det, chunk=16384, use_checkpoint=True):
        """theta (V,) radians (angle_sign already applied), det (D,) -> (V,D).

        chunk: Gaussians per chunk. With use_checkpoint=True every chunk is
        recomputed in the backward pass, so peak memory is bounded by
        V * chunk * D floats instead of V * N * D.
        TODO(tune): lower `chunk` (or views_per_step) if you hit CUDA OOM.
        """
        args = (self.xy, self.log_scale, self.rot, self.log_rho)
        if chunk is None or chunk >= self.n:
            return _project_chunk(*args, theta, det, self.s_min, self.s_max)
        out = 0.0
        for i in range(0, self.n, chunk):
            sl = slice(i, i + chunk)
            a = tuple(p[sl] for p in args)
            if use_checkpoint and torch.is_grad_enabled():
                out = out + checkpoint(_project_chunk, *a, theta, det,
                                       self.s_min, self.s_max, use_reentrant=False)
            else:
                out = out + _project_chunk(*a, theta, det, self.s_min, self.s_max)
        return out

    # -- image-domain rendering (evaluation + TV regulariser) -----------------
    def render_region(self, ys, xs, chunk=64):
        """Evaluate the density on the grid ys (H,) x xs (W,) (pixel offsets from
        the centre, float tensors). Differentiable. Returns (H, W)."""
        gy, gx = torch.meshgrid(ys, xs, indexing="ij")
        gx, gy = gx.reshape(-1), gy.reshape(-1)
        s, rho = self.scales(), self.rho()
        cp, sp = torch.cos(self.rot), torch.sin(self.rot)
        out = torch.zeros_like(gx)
        for i in range(0, self.n, chunk):
            sl = slice(i, i + chunk)
            dx = gx[None, :] - self.xy[sl, 0:1]                      # (c,P)
            dy = gy[None, :] - self.xy[sl, 1:2]
            a = (dx * cp[sl, None] + dy * sp[sl, None]) / s[sl, 0:1]   # along e1
            b = (-dx * sp[sl, None] + dy * cp[sl, None]) / s[sl, 1:2]  # along e2
            out = out + (rho[sl, None] * torch.exp(-0.5 * (a * a + b * b))).sum(0)
        inside = (gx ** 2 + gy ** 2) <= self.c ** 2   # same circle as iradon(circle=True)
        return (out * inside).view(len(ys), len(xs))

    @torch.no_grad()
    def render_image(self, chunk=64):
        """Full size x size reconstruction in the model's (normalised) units."""
        dev = self.xy.device
        co = torch.arange(self.size, device=dev, dtype=torch.float32) - self.c
        return self.render_region(co, co, chunk=chunk)

    # -- adaptive density control (clone / split / prune) ---------------------
    def _set(self, xy, log_scale, rot, log_rho):
        self.xy = nn.Parameter(xy.contiguous())
        self.log_scale = nn.Parameter(log_scale.contiguous())
        self.rot = nn.Parameter(rot.contiguous())
        self.log_rho = nn.Parameter(log_rho.contiguous())

    @torch.no_grad()
    def densify_and_prune(self, grad_accum, denom, top_frac, split_px, rho_prune, max_n):
        """Standard 3DGS-style control, simplified for 2D CT.

        * candidates = the `top_frac` fraction of Gaussians with the largest mean
          positional-gradient norm (RELATIVE threshold, so it does not depend on
          the loss scale). TODO(tune): switch to an absolute threshold if wanted.
        * candidate with max-scale <= split_px -> CLONE (density halved on both copies)
        * candidate with max-scale  > split_px -> SPLIT into 2 children sampled
          from the parent, scale/1.6, density x1.28 (total mass is preserved)
        * afterwards: prune Gaussians with density < rho_prune.
        Net growth per call is at most `top_frac*N`, capped by `max_n`.
        """
        n, dev = self.n, self.xy.device
        avg = grad_accum / denom.clamp(min=1.0)
        s = self.scales()
        budget = int(min(top_frac * n, max(0, max_n - n)))
        cand = torch.zeros(n, dtype=torch.bool, device=dev)
        if budget > 0:
            cand[torch.topk(avg, budget).indices] = True
        big = s.max(dim=1).values > split_px
        clone_m, split_m = cand & ~big, cand & big

        xy, ls = self.xy.data, self.log_scale.data
        rot, lr_ = self.rot.data, self.log_rho.data.clone()
        lr_[clone_m] -= math.log(2.0)          # clone: halve density of both copies

        def offset(m, factor):                 # sample a point from each Gaussian
            z = torch.randn(int(m.sum()), 2, device=dev) * s[m] * factor
            c, sn = torch.cos(rot[m]), torch.sin(rot[m])
            return torch.stack([z[:, 0] * c - z[:, 1] * sn, z[:, 0] * sn + z[:, 1] * c], 1)

        parts_xy = [xy[~split_m], xy[clone_m] + offset(clone_m, 0.5)]
        parts_ls = [ls[~split_m], ls[clone_m]]
        parts_rot = [rot[~split_m], rot[clone_m]]
        parts_lr = [lr_[~split_m], lr_[clone_m]]
        for _ in range(2):                     # two children per split Gaussian
            parts_xy.append(xy[split_m] + offset(split_m, 1.0))
            parts_ls.append(ls[split_m] - math.log(1.6))
            parts_rot.append(rot[split_m])
            parts_lr.append(lr_[split_m] + math.log(1.6 ** 2 / 2.0))

        new_xy, new_ls = torch.cat(parts_xy), torch.cat(parts_ls)
        new_rot, new_lr = torch.cat(parts_rot), torch.cat(parts_lr)
        keep = torch.exp(new_lr) > rho_prune
        if keep.sum() < 100:                   # safety net
            keep[:] = True
        self._set(new_xy[keep], new_ls[keep], new_rot[keep], new_lr[keep])
        return dict(cloned=int(clone_m.sum()), split=int(split_m.sum()),
                    pruned=int((~keep).sum()), n=self.n)

### Initialisation
R2-Gaussian style: sample the Gaussian centres from the **sparse-view FBP** (which only uses the same sparse data the other methods get). `init="random"` is available as an ablation for the report.

In [ ]:
def density_reference(img, size=888, pct=99.5):
    """A robust 'tissue density' scale of an image (used to make the TV term and
    the pruning threshold independent of the sinogram scale)."""
    c = size // 2
    yy, xx = np.mgrid[:size, :size] - c
    inside = (xx ** 2 + yy ** 2) <= c ** 2
    v = img[inside & (img > 0)]
    return float(np.percentile(v, pct)) if v.size else 1.0


def init_from_image(img, n, size=888, uniform_frac=0.1, gamma=0.5, seed=0, s_min=0.5, s_max=60.0):
    """Sample Gaussian centres from a (sparse-view FBP) image - R2-Gaussian style.

    img : 2D float array in the units the model will be trained in.
    Points are drawn with probability ~ max(img,0)^gamma inside the head
    (gamma<1 flattens the bone/soft-tissue contrast so soft tissue gets points too).
    `uniform_frac` of the points are scattered uniformly in the circle so that
    streak artefacts in air can still be 'explained' (and pruned if useless).
    TODO(tune): n, gamma, uniform_frac, 5% threshold below.
    """
    rng = np.random.default_rng(seed)
    S, c = size, size // 2
    yy, xx = np.mgrid[:S, :S] - c
    inside = (xx ** 2 + yy ** 2) <= c ** 2
    pos = np.clip(img, 0, None) * inside
    pos = np.where(pos > 0.05 * pos.max(), pos, 0.0)        # drop air / streak noise
    area = max(int((pos > 0).sum()), 1)

    n_f = int(round(n * (1 - uniform_frac)))
    n_u = n - n_f
    w = pos.ravel() ** gamma
    w /= w.sum()
    flat = rng.choice(S * S, size=n_f, replace=True, p=w)
    py, px = np.unravel_index(flat, (S, S))
    x_f = px - c + rng.uniform(-0.5, 0.5, n_f)
    y_f = py - c + rng.uniform(-0.5, 0.5, n_f)
    r = 0.95 * c * np.sqrt(rng.uniform(size=n_u))
    a = TWO_PI * rng.uniform(size=n_u)
    x = np.concatenate([x_f, r * np.cos(a)])
    y = np.concatenate([y_f, r * np.sin(a)])

    ix = np.clip(np.round(x + c).astype(int), 0, S - 1)
    iy = np.clip(np.round(y + c).astype(int), 0, S - 1)
    val = np.clip(img[iy, ix], 0.01 * img.max(), None)      # density floor = 1% of peak
    a2 = area / n                                            # image area per Gaussian
    s0 = float(np.clip(0.4 * np.sqrt(a2), 1.0, 6.0))         # initial isotropic scale
    rho0 = val * a2 / (TWO_PI * s0 ** 2)                     # ~ val when s0 = 0.4*sqrt(a2)

    t = lambda v: torch.tensor(v, dtype=torch.float32)
    return GaussianCT(
        xy=t(np.stack([x, y], 1)),
        log_scale=torch.full((n, 2), math.log(s0)),
        rot=t(rng.uniform(0, math.pi, n)),
        log_rho=t(np.log(rho0)),
        size=size, s_min=s_min, s_max=s_max,
    )


def init_random(n, size=888, seed=0, s0=3.0, rho0=1e-3, s_min=0.5, s_max=60.0):
    """Ablation: uniform random initialisation inside the reconstruction circle.
    Density is calibrated afterwards by train_3dgs.calibrate_density()."""
    rng = np.random.default_rng(seed)
    c = size // 2
    r = 0.95 * c * np.sqrt(rng.uniform(size=n))
    a = TWO_PI * rng.uniform(size=n)
    xy = np.stack([r * np.cos(a), r * np.sin(a)], 1)
    t = lambda v: torch.tensor(v, dtype=torch.float32)
    return GaussianCT(
        xy=t(xy),
        log_scale=torch.full((n, 2), math.log(s0)),
        rot=t(rng.uniform(0, math.pi, n)),
        log_rho=torch.full((n,), math.log(rho0)),
        size=size, s_min=s_min, s_max=s_max,
    )

## 4. Training functions
- Sinogram is divided by its maximum before optimisation (so values are ~[0, 1]); the rendered image is multiplied back, so the saved result is in **raw FBP units**.
- Loss = L1 on the projections of a random subset of `views_per_step` training angles (+ optional L2) + `tv_weight` × TV of a random rendered image patch.
- Densification (clone / split / prune) between 10 % and 60 % of the iterations; learning rates decay exponentially.
- After fitting, projections at **held-out angles** (unique half-circle angles not used for fitting) are rendered and compared with the true sinogram → view-synthesis metrics `proj_psnr / proj_ssim / proj_rmse`.
- `runtime_s` = FBP init + fitting + final render (comparable to the FBP runtime); evaluation is not counted.

In [ ]:
def make_optimizer(model, cfg):
    groups = [
        {"params": [model.xy], "lr": cfg.lr_xy},
        {"params": [model.log_scale], "lr": cfg.lr_scale},
        {"params": [model.rot], "lr": cfg.lr_rot},
        {"params": [model.log_rho], "lr": cfg.lr_rho},
    ]
    for g in groups:
        g["base_lr"] = g["lr"]
    return torch.optim.Adam(groups, eps=1e-15)


@torch.no_grad()
def calibrate_density(model, theta, det, target, cfg):
    """Scale all densities so the initial projections have the same total mass as the
    measured sparse sinogram (cheap fix for the init-density approximation)."""
    pred_sum = 0.0
    for i in range(0, len(theta), 16):
        pred_sum += float(model.project(theta[i:i + 16], det, cfg.gauss_chunk, False).sum())
    alpha = float(target.sum()) / max(pred_sum, 1e-12)
    model.log_rho.data += float(np.log(max(alpha, 1e-6)))
    return alpha


def tv_patch_loss(model, patch, rho_ref, chunk):
    """Total variation of a random patch of the rendered image (differentiable).
    Normalised by rho_ref so the weight does not depend on the sinogram scale."""
    c = model.c
    r = 0.8 * c * np.sqrt(np.random.rand())
    a = 2 * np.pi * np.random.rand()
    cx, cy = int(r * np.cos(a)), int(r * np.sin(a))
    dev = model.xy.device
    ar = torch.arange(patch, device=dev, dtype=torch.float32) - patch // 2
    img = model.render_region(ar + cy, ar + cx, chunk=chunk) / rho_ref
    return (img[1:] - img[:-1]).abs().mean() + (img[:, 1:] - img[:, :-1]).abs().mean()

In [ ]:
def fit_one(sino, n_views, cfg, device, preview_path=None, gt=None):
    """Fit one slice. sino: raw (N_DET, N_ANGLES). Returns dict with reconstruction etc."""
    set_seed(cfg.seed)
    sync = torch.cuda.synchronize if device.type == "cuda" else (lambda: None)
    sync()
    t0 = time.perf_counter()

    idx = view_indices(n_views)
    theta_deg = THETA_FULL_DEG[idx]
    sparse = sino[:, idx]                              # (N_DET, V)
    scale = float(sparse.max()) or 1.0                 # normalise sinogram to ~[0,1] for optimisation
    fbp = iradon(sparse, theta=theta_deg, filter_name=FILTER,
                 output_size=OUTPUT_SIZE, circle=True).astype(np.float32)
    fbp_n = fbp / scale
    rho_ref = density_reference(fbp_n, OUTPUT_SIZE)

    if cfg.init == "fbp":
        model = init_from_image(fbp_n, cfg.n_init, OUTPUT_SIZE, cfg.init_uniform_frac,
                                cfg.init_gamma, cfg.seed, cfg.s_min, cfg.s_max)
    else:
        model = init_random(cfg.n_init, OUTPUT_SIZE, cfg.seed, s_min=cfg.s_min, s_max=cfg.s_max)
    model = model.to(device)

    det = torch.arange(N_DET, device=device, dtype=torch.float32) - N_DET // 2
    theta = torch.tensor(np.deg2rad(theta_deg) * cfg.angle_sign, dtype=torch.float32, device=device)
    target = torch.tensor((sparse / scale).T, dtype=torch.float32, device=device)  # (V, D)
    alpha = calibrate_density(model, theta, det, target, cfg)

    opt = make_optimizer(model, cfg)
    V = min(cfg.views_per_step, n_views)
    d_from, d_until = int(cfg.densify_from * cfg.iters), int(cfg.densify_until * cfg.iters)
    gacc, gden = torch.zeros(model.n, device=device), torch.zeros(model.n, device=device)
    loss_val, t_loop = float("nan"), time.perf_counter()

    for it in range(1, cfg.iters + 1):
        decay = cfg.lr_final_ratio ** (it / cfg.iters)
        for g in opt.param_groups:
            g["lr"] = g["base_lr"] * decay

        sel = torch.randperm(n_views, device=device)[:V]
        pred = model.project(theta[sel], det, cfg.gauss_chunk, cfg.use_checkpoint)
        err = pred - target[sel]
        loss = err.abs().mean()
        if cfg.l2_weight > 0:
            loss = loss + cfg.l2_weight * (err ** 2).mean()
        if cfg.tv_weight > 0:
            loss = loss + cfg.tv_weight * tv_patch_loss(model, cfg.tv_patch, rho_ref, cfg.tv_chunk)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        in_window = d_from <= it <= d_until
        if in_window and model.xy.grad is not None:
            gacc += model.xy.grad.norm(dim=-1)
            gden += 1
        opt.step()
        with torch.no_grad():
            model.xy.data.clamp_(-model.c, model.c)

        if in_window and it % cfg.densify_every == 0:
            info = model.densify_and_prune(gacc, gden, cfg.densify_top_frac, cfg.split_px,
                                           cfg.prune_rho_rel * rho_ref, cfg.max_gaussians)
            opt = make_optimizer(model, cfg)   # TODO(improve): keep Adam state instead of resetting
            gacc, gden = torch.zeros(model.n, device=device), torch.zeros(model.n, device=device)
            print(f"    it {it}: densify {info}")

        if it % cfg.log_every == 0 or it == cfg.iters:
            loss_val = float(loss)
            print(f"    it {it:5d}/{cfg.iters} | loss {loss_val:.5f} | N={model.n} | "
                  f"{(time.perf_counter() - t_loop) / it:.3f} s/it")

    rec_n = model.render_image(chunk=64)
    rec = rec_n.cpu().numpy().astype(np.float32) * scale     # back to RAW FBP units
    sync()
    runtime = time.perf_counter() - t0                       # fit + render (comparable to FBP runtime)

    out = dict(rec=rec, runtime=runtime, n_gauss=model.n, loss=loss_val, alpha=alpha, fbp=fbp)

    # ---- view synthesis: held-out angles (unique half-circle indices not used for fitting) ----
    n_half = N_ANGLES // 2 if ANGLE_RANGE == 360 else N_ANGLES
    held = np.setdiff1d(np.arange(n_half), idx)
    if len(held) > 0:
        held = held[np.unique(np.round(np.linspace(0, len(held) - 1, cfg.n_heldout)).astype(int))]
        th_h = torch.tensor(np.deg2rad(THETA_FULL_DEG[held]) * cfg.angle_sign,
                            dtype=torch.float32, device=device)
        with torch.no_grad():
            cols = [model.project(th_h[i:i + 16], det, cfg.gauss_chunk, False)
                    for i in range(0, len(th_h), 16)]
        pred_sino = torch.cat(cols).cpu().numpy().T * scale   # (D, n_held) raw units
        gt_sino = sino[:, held]
        g_n, p_n = metrics.normalise_pair(gt_sino, pred_sino)  # no head mask in sinogram domain
        out.update(proj_psnr=metrics.psnr(p_n, g_n), proj_ssim=metrics.ssim(p_n, g_n),
                   proj_rmse=metrics.rmse(p_n, g_n))

    if preview_path is not None and gt is not None:
        save_preview(preview_path, gt, fbp, rec)

    del model, opt
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return out

In [ ]:
def save_preview(path, gt, fbp, rec):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    _, f_n = metrics.normalise_pair(gt, fbp)
    g_n, r_n = metrics.normalise_pair(gt, rec)
    fig, ax = plt.subplots(1, 4, figsize=(16, 4.2))
    for a, im, t in zip(ax[:3], [f_n, r_n, g_n], ["sparse FBP", "3DGS", "ground truth (full FBP)"]):
        a.imshow(im, cmap="gray", vmin=0, vmax=1); a.set_title(t); a.axis("off")
    ax[3].imshow(np.abs(r_n - g_n), cmap="magma", vmin=0, vmax=0.3); ax[3].set_title("|3DGS - GT|")
    ax[3].axis("off")
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout(); plt.savefig(path, dpi=110); plt.close(fig)

### Main loop
Same structure as the baseline: series → slices (every `SLICE_STEP`-th) → view counts. A result row is appended to the CSV after **every fit** and finished fits are skipped, so a Colab disconnect only costs the current fit – just run the cell again.

In [ ]:
def main(cfg):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cpu":
        print("[warn] no GPU found - this will be extremely slow. TODO(run): use a GPU runtime.")
    out_dir = Path(cfg.out_dir)
    csv_name = "3dgs_results.csv"
    tag = cfg.tag or ("smoke" if cfg.smoke else "")
    if tag:   # smoke / tuning runs never touch the official results
        out_dir, csv_name = out_dir / f"_{tag}", f"3dgs_{tag}.csv"
    out_dir.mkdir(parents=True, exist_ok=True)
    csv = out_dir / csv_name

    files, sdir = find_sinograms(cfg)
    if cfg.smoke:
        files = files[:1]
    done = set()
    if csv.exists():
        d = pd.read_csv(csv)
        done = set(zip(d["series"], d["slice"], d["views"]))

    for f in files:
        sid = series_id(f, sdir)
        sinos = as_slices(np.load(f))
        slices = list(range(0, sinos.shape[0], cfg.slice_step))
        if cfg.smoke:
            slices = slices[:1]
        elif cfg.max_slices:
            slices = slices[:cfg.max_slices]
        for s in slices:
            gt = None
            for v in cfg.views:
                if (sid, s, v) in done:
                    print(f"{sid} slice {s:02d} v{v:03d}: already done, skipped"); continue
                gt = load_gt(cfg, sid, s, sinos[s]) if gt is None else gt
                print(f"{sid} slice {s:02d} | {v} views")
                prev = out_dir / "previews" / f"{sid}_slice{s:02d}_v{v:03d}.png" if cfg.save_preview else None
                r = fit_one(sinos[s], v, cfg, device, prev, gt)
                np.save(out_dir / f"{sid}_slice{s:02d}_v{v:03d}.npy", r["rec"])
                m = metrics.evaluate(r["rec"], gt)
                m_fbp = metrics.evaluate(r["fbp"], gt)
                row = {"series": sid, "slice": s, "method": "3DGS", "views": v,
                       "runtime_s": r["runtime"], **m,
                       "proj_psnr": r.get("proj_psnr"), "proj_ssim": r.get("proj_ssim"),
                       "proj_rmse": r.get("proj_rmse"), "n_gaussians": r["n_gauss"],
                       "final_loss": r["loss"], "iters": cfg.iters, "seed": cfg.seed,
                       "fbp_psnr_same_views": m_fbp["psnr"], "fbp_ssim_same_views": m_fbp["ssim"]}
                pd.DataFrame([row]).to_csv(csv, mode="a", header=not csv.exists(), index=False)
                print(f"  -> PSNR {m['psnr']:.2f} (FBP {m_fbp['psnr']:.2f}) | SSIM {m['ssim']:.3f} "
                      f"(FBP {m_fbp['ssim']:.3f}) | head-PSNR {m['psnr_head']:.2f} | "
                      f"held-out proj PSNR {r.get('proj_psnr', float('nan')):.2f} | {r['runtime']:.0f} s")

    if csv.exists():
        res = pd.read_csv(csv)
        cols = ["psnr", "ssim", "rmse", "psnr_head", "ssim_head", "rmse_head",
                "proj_psnr", "proj_ssim", "runtime_s"]
        summary = res.groupby("views")[cols].agg(["mean", "std"]).sort_index(ascending=False).round(3)
        summary.to_csv(out_dir / csv_name.replace(".csv", "_summary.csv"))
        print(summary)

## 5. Geometry check  (run this first!)
1. Our analytic projector vs. `skimage.radon` of the rendered image of 3 test Gaussians → relative error should be well below 5 %, and it tells which `angle_sign` is right.
2. Real data: `radon(ground-truth image)` vs. the measured sinogram → the **best-fit scale must be ≈ 1.0**. If it is not, image units ≠ sinogram units and everything afterwards would be wrong → stop and fix first.

In [ ]:
def sanity_check(cfg, device):
    S, c = N_DET, N_DET // 2
    t = lambda v: torch.tensor(v, dtype=torch.float32)
    model = GaussianCT(t([[0, 0], [60, -40], [-100, 80]]),
                       t([[10, 5], [4, 12], [15, 15]]).log(), t([0.3, 1.0, 0.0]),
                       t([1.0, 2.0, 0.5]).log(), size=S).to(device)
    img = model.render_image(chunk=3).cpu().numpy()
    theta_deg = np.array([0, 17, 45, 90, 133, 171], dtype=float)
    ref = radon(img, theta=theta_deg, circle=True)             # skimage forward projector
    det = torch.arange(S, dtype=torch.float32, device=device) - c
    print("[1] Gaussian projector vs skimage.radon of the rendered image")
    errs = {}
    for sign in (1.0, -1.0):
        th = torch.tensor(np.deg2rad(theta_deg) * sign, dtype=torch.float32, device=device)
        p = model.project(th, det, chunk=None).detach().cpu().numpy().T
        errs[sign] = np.linalg.norm(p - ref) / np.linalg.norm(ref)
        print(f"    angle_sign {sign:+.0f}: relative error {errs[sign]:.3%}")
    best = min(errs, key=errs.get)
    print(f"    -> use angle_sign = {best:+.0f}" + ("" if errs[best] < 0.05 else
          "   [WARNING: error still > 5% - the projector/renderer geometry is off, do not run!]"))

    print("[2] real data: radon(ground-truth image) vs measured sinogram (needs data paths)")
    try:
        cfg2 = Cfg(**{**cfg.__dict__, "series": cfg.series[:1] if cfg.series else None})
        files, sdir = find_sinograms(cfg2)
        sino = as_slices(np.load(files[0]))[0]
        sid = series_id(files[0], sdir)
        gt = load_gt(cfg, sid, 0, sino)
        idx = view_indices(16)
        q = radon(gt, theta=THETA_FULL_DEG[idx], circle=True)
        m = sino[:, idx]
        a = float((q * m).sum() / (q * q).sum())
        rel = np.linalg.norm(a * q - m) / np.linalg.norm(m)
        print(f"    best-fit scale {a:.3f} (should be ~1.0), residual after fit {rel:.1%}")
    except Exception as e:  # noqa: BLE001
        print(f"    skipped ({type(e).__name__}: {e})")

In [ ]:
sanity_check(cfg, device)

**Result of check 1 (first run, Windows/local):** `angle_sign +1` → 0.077 % error, `-1` → 108.5 % → keep `angle_sign = +1`.
`TODO(run)`: check 2 has not been run on real data yet – note the printed best-fit scale here: `___`

## 6. Smoke test (1 slice, 49 views, 300 iterations)
Writes to `<out_dir>/_smoke/` only (never to the official results). Check:
- **s/it** in the log → estimate the full runtime = *number of fits × iterations × s/it* (the default run is 2 series × 11 slices × 4 view counts = 88 fits × 3000 iterations).
- the preview PNG in `<out_dir>/_smoke/previews/`: 3DGS PSNR should be ≥ the FBP PSNR at the same views (`fbp_psnr_same_views` in the CSV).
- CUDA out of memory → lower `views_per_step` and/or `gauss_chunk`.

In [ ]:
smoke_cfg = replace(cfg, smoke=True, views=[49], iters=300, n_init=5000,
                    log_every=50, densify_every=50)
main(smoke_cfg)

## 7. Tuning on the validation series
README: hyper-parameters are tuned on the **validation** set, final scores are computed on the test set → never tune on `cfg.series`.
`series_1023` is the validation series in Jule's split (`TODO(run)`: confirm with the team). Every run gets its own `tag`, so outputs go to `<out_dir>/_<tag>/` and cannot mix with the official results.
Change one thing at a time, compare the summary tables, then write the winning values into the `Cfg` defaults (§1) and into `Log/log_gianne.md`.
Knobs by importance: `iters`, `n_init`, `lr_xy`, `lr_rho`, `tv_weight` {0, 0.005, 0.01, 0.05}, `densify_*`, `split_px`, `init` ("fbp" vs "random").

In [ ]:
tune_cfg = replace(cfg, series=["series_1023"], max_slices=3, views=[49, 98],
                   tag="tune_base")        # TODO(tune): new tag for every variant
# e.g. replace(cfg, ..., tv_weight=0.05, tag="tune_tv005")  /  iters=5000, tag="tune_it5000"
main(tune_cfg)

## 8. Full run
Reconstructions go to `<out_dir>/series_XXXX_sliceYY_vNNN.npy` (888×888 float32, **raw FBP units**) – the filename pattern Sophia asked for.
`TODO(run)`: Sophia needs the files in Drive `outputs/<method>/` by 20.10 – confirm the folder name. Resume-safe: finished (series, slice, views) rows in the CSV are skipped.

In [ ]:
main(cfg)

## 9. Summary: mean ± SD per view count

In [ ]:
CSV = Path(cfg.out_dir) / "3dgs_results.csv"
res = pd.read_csv(CSV)
cols = ["psnr", "ssim", "rmse", "psnr_head", "ssim_head", "rmse_head",
        "proj_psnr", "proj_ssim", "runtime_s"]
summary = res.groupby("views")[cols].agg(["mean", "std"]).sort_index(ascending=False).round(3)
summary.to_csv(Path(cfg.out_dir) / "3dgs_results_summary.csv")
print(f"{res.series.nunique()} series, {len(res) // res.views.nunique()} slices per view count, "
      f"median {res.n_gaussians.median():.0f} Gaussians")
summary

In [ ]:
# 3DGS vs. plain FBP with the SAME sparse views (FBP computed during our init -> same slices, fair comparison)
g = res.groupby("views")
x = sorted(res.views.unique())
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, (m_3dgs, m_fbp, label) in zip(ax[:2], [("psnr", "fbp_psnr_same_views", "PSNR [dB]"),
                                              ("ssim", "fbp_ssim_same_views", "SSIM")]):
    a.errorbar(x, g[m_3dgs].mean()[x], yerr=g[m_3dgs].std()[x], fmt="o-", capsize=3, label="3DGS")
    a.errorbar(x, g[m_fbp].mean()[x], yerr=g[m_fbp].std()[x], fmt="s--", capsize=3, label="FBP (same views)")
    a.set_xscale("log"); a.set_xticks(x); a.set_xticklabels(x)
    a.set_xlabel("number of views"); a.set_ylabel(label); a.grid(alpha=.3); a.legend()
a = ax[2]   # view synthesis: quality of projections at angles never used for fitting
a.errorbar(x, g["proj_psnr"].mean()[x], yerr=g["proj_psnr"].std()[x], fmt="o-", capsize=3, color="C2")
a.set_xscale("log"); a.set_xticks(x); a.set_xticklabels(x)
a.set_xlabel("number of training views"); a.set_ylabel("held-out projection PSNR [dB]"); a.grid(alpha=.3)
a.set_title("view synthesis (unseen angles)")
plt.tight_layout(); plt.show()

## 10. One slice at all view counts (figure for the report)

In [ ]:
row0 = res.iloc[0]
sid, s = row0.series, int(row0.slice)
gt = np.load(Path(cfg.gt_dir) / f"{sid}_slice{s:02d}.npy")
vs = sorted(res[(res.series == sid) & (res.slice == s)].views.unique(), reverse=True)

fig, ax = plt.subplots(1, len(vs) + 1, figsize=(3.2 * (len(vs) + 1), 3.6))
gt_n = metrics.normalise_pair(gt, gt)[0]
ax[0].imshow(gt_n, cmap="gray", vmin=0, vmax=1); ax[0].set_title("ground truth (984 views)")
for a, v in zip(ax[1:], vs):
    rec = np.load(Path(cfg.out_dir) / f"{sid}_slice{s:02d}_v{v:03d}.npy")
    r_n = metrics.normalise_pair(gt, rec)[1]
    p = res[(res.series == sid) & (res.slice == s) & (res.views == v)].psnr.iloc[0]
    a.imshow(r_n, cmap="gray", vmin=0, vmax=1); a.set_title(f"3DGS, {v} views\nPSNR {p:.1f} dB")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## Notes for the log
- Per-scan optimisation: one Gaussian set per slice, no training data; the ground truth (full-view FBP, Sophia's cached `.npy`) is used only for evaluation.
- Sparse views = the baseline's `view_indices` (sampled within 180°); sinogram normalised by its max for optimisation, output rescaled to raw FBP units.
- Initialised from the sparse-view FBP (same information as the other methods get); `init="random"` ablation possible.
- View-synthesis metrics use held-out angles from the first 180° only (θ and θ+180° are redundant in parallel beam).
- `runtime_s` includes FBP init + optimisation + render; compare with FBP (~seconds) and U-Net (single forward pass after training) in the report.
- Limitations to mention: 2D slice-wise model (no 3D consistency across slices); dense projector is slow; hyper-parameters tuned on one validation series only.